# 4 — Basic ReAct Agent (`create_react_agent`)

The fastest way to build a tool-calling agent in LangGraph: one function call wires LLM + tools + memory into a runnable graph.

**What you'll learn**
- `create_react_agent(model, tools, prompt)` — the prebuilt that handles the ReAct loop automatically
- Tool selection: how the LLM picks the right tool based on docstrings
- Multi-tool calls: chaining tools in a single query
- `MemorySaver` + `thread_id` — multi-turn conversation with persistent memory
- `stream_mode="values"` — printing the latest message after each step
- When to use `create_react_agent` vs building a graph manually

**Companion examples:** 1-basic-local-rag (manual graph vs prebuilt), 30-agentic-rag (create_react_agent with retrieval tools)

In [ ]:
# ── 1. Install dependencies (Colab only) ──────────────────────────────────────
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai python-dotenv langgraph

In [ ]:
# ── 2. API key ─────────────────────────────────────────────────────────────────
import os

from dotenv import load_dotenv

if not IN_COLAB:
    load_dotenv()

if IN_COLAB and not os.environ.get("OPENAI_API_KEY"):
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# ── 3. Define tools and build the agent ───────────────────────────────────────
# @tool converts a function into a LangChain tool.
# The DOCSTRING is what the LLM reads to decide WHEN to call each tool.
# The SIGNATURE defines the JSON schema the LLM uses to call it.
#
# create_react_agent handles the full ReAct loop automatically:
#   LLM decides which tool -> tool runs -> result added to messages -> repeat
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent


@tool
def find_sum(x: int, y: int) -> int:
    """Add two numbers and return their sum. Takes two integers as inputs."""
    return x + y


@tool
def find_product(x: int, y: int) -> int:
    """Multiply two numbers and return their product. Takes two integers as inputs."""
    return x * y


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

system_prompt = SystemMessage(
    "You are a Math assistant. Solve problems using ONLY the available tools. "
    "Do NOT compute answers yourself — always use a tool."
)

agent = create_react_agent(
    model=llm,
    tools=[find_sum, find_product],
    prompt=system_prompt,
    checkpointer=MemorySaver(),
)

print("Agent ready — tools: find_sum, find_product")

In [ ]:
# ── 4. Single tool call with streaming ────────────────────────────────────────
# stream_mode="values" yields the full state after each step.
# Printing the last message each step shows the tool-call sequence clearly.
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "math-demo"}}

print("Q: What is the sum of 2 and 3?\n")
for step in agent.stream(
    {"messages": [HumanMessage("What is the sum of 2 and 3?")]},
    config,
    stream_mode="values",
):
    msg = step["messages"][-1]
    print(f"[{type(msg).__name__}] {str(msg.content)[:100]}")

In [ ]:
# ── 5. Multi-step tool chaining ───────────────────────────────────────────────
# The agent chains tools: find_product(3, 2) -> find_sum(result, 5).
# Watch the message trace to see each tool call and its return value.
config2 = {"configurable": {"thread_id": "math-chain"}}

print("Q: What is 3 multiplied by 2, then add 5?\n")
result = agent.invoke(
    {"messages": [HumanMessage("What is 3 multiplied by 2, then add 5?")]},
    config2,
)
for msg in result["messages"]:
    role = type(msg).__name__
    content = str(msg.content)[:120] if msg.content else ""
    print(f"[{role}] {content}")

In [ ]:
# ── 6. Multi-turn memory with thread_id ───────────────────────────────────────
# Same thread_id = same MemorySaver checkpoint = the agent remembers previous turns.
# Different thread_id = fresh conversation with no prior context.
config3 = {"configurable": {"thread_id": "math-memory"}}

r1 = agent.invoke({"messages": [HumanMessage("What is 7 plus 8?")]}, config3)
print("Turn 1:", r1["messages"][-1].content)

r2 = agent.invoke({"messages": [HumanMessage("Multiply that result by 3.")]}, config3)
print("Turn 2:", r2["messages"][-1].content)

# Fresh thread — no memory of turn 1
config4 = {"configurable": {"thread_id": "math-fresh"}}
r3 = agent.invoke({"messages": [HumanMessage("Multiply that result by 3.")]}, config4)
print("Fresh thread:", r3["messages"][-1].content)

## Exercises

**Exercise 1 — Add a division tool:** Create `@tool divide(x: int, y: int) -> float`. Add it and ask `"What is 100 divided by 4, then times 3?"` — does the agent chain the two tools correctly?

**Exercise 2 — Docstring matters:** Change `find_sum`'s docstring to `"Look up weather"`. Ask `"What is 5 + 3?"`. Does the agent still pick the right tool? What does this tell you about how tool selection works?

**Exercise 3 — Without memory:** Remove `checkpointer=MemorySaver()` from `create_react_agent`. Try the multi-turn test from cell 6. Does the agent remember the first result?

**Exercise 4 — Compare manual vs prebuilt:** Look at example 1 (basic-local-rag). It builds the same `agent → tools` cycle manually with `StateGraph`. `create_react_agent` is that graph but pre-assembled. What flexibility do you lose?

## What's next

- **1-basic-local-rag** — build the same loop manually to understand the internals
- **30-agentic-rag** — `create_react_agent` with retrieval + web search + calculator tools
- **5-react-agent-lg** — two-agent critic loop: summarizer and reviewer checking each other's work